In [ ]:
#####Notebook 03 takes these raw zarr arrays and prepares them for VegFormer:

Patch extraction — cuts the 377×313 spatial grid into 64×64 pixel patches with overlap
Temporal alignment — aligns S2 16-day and CHIRPS 16-day to the same timestep index
Normalization — per-band z-score normalization using training set statistics
Dataset split — train / val / test split stratified by vegetation zone
PyTorch Dataset — wraps everything into a DataLoader ready for VegFormer

In [1]:
import zarr
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path('C:/vegformer/data/processed')

datasets = {
    'ndvi':    'ndvi_monthly_2019_2024_clean.zarr',
    's2':      's2_16day_2019_2024.zarr',
    's1':      's1_16day_2019_2024.zarr',
    'chirps':  'chirps_16day_2019_2024.zarr',
}

print("Checking all inputs...")
for name, fname in datasets.items():
    z    = zarr.open(str(PROCESSED_DIR / fname), mode='r')
    nans = np.isnan(z[:]).sum()
    print(f"  {name:<8} shape={str(z.shape):<28} NaNs={nans}")

print("\nAll inputs ready — proceeding to patch extraction")

Checking all inputs...
  ndvi     shape=(72, 377, 313)               NaNs=0
  s2       shape=(137, 14, 377, 313)          NaNs=0
  s1       shape=(137, 2, 377, 313)           NaNs=0
  chirps   shape=(137, 4, 19, 16)             NaNs=0

All inputs ready — proceeding to patch extraction


In [ ]:
### Full environment for notebook 03:

In [2]:
import zarr
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import json

PROCESSED_DIR = Path('C:/vegformer/data/processed')
PATCHES_DIR   = Path('C:/vegformer/data/patches')
PATCHES_DIR.mkdir(parents=True, exist_ok=True)

# Load all arrays into memory
print("Loading arrays...")
s2_z      = zarr.open(str(PROCESSED_DIR / 's2_16day_2019_2024.zarr'),           mode='r')
s1_z      = zarr.open(str(PROCESSED_DIR / 's1_16day_2019_2024.zarr'),           mode='r')
chirps_z  = zarr.open(str(PROCESSED_DIR / 'chirps_16day_2019_2024.zarr'),       mode='r')
ndvi_z    = zarr.open(str(PROCESSED_DIR / 'ndvi_monthly_2019_2024_clean.zarr'), mode='r')

s2_arr     = s2_z[:]      # (137, 14, 377, 313)
s1_arr     = s1_z[:]      # (137,  2, 377, 313)
ndvi_arr   = ndvi_z[:]    # ( 72, 377, 313)

s2_dates   = list(s2_z.attrs['dates'])
s1_dates   = list(s1_z.attrs['dates'])
ndvi_dates = list(ndvi_z.attrs['dates'])

print(f"S2:    {s2_arr.shape}")
print(f"S1:    {s1_arr.shape}")
print(f"NDVI:  {ndvi_arr.shape}")
print(f"CHIRPS loaded lazily from zarr (small — 19x16)")
print("\nAll arrays loaded")

Loading arrays...
S2:    (137, 14, 377, 313)
S1:    (137, 2, 377, 313)
NDVI:  (72, 377, 313)
CHIRPS loaded lazily from zarr (small — 19x16)

All arrays loaded


In [ ]:
#####Align the temporal dimensions. S2 and S1 are on 16-day composites, NDVI is monthly. Need them on the same timestep index for VegFormer

In [7]:
aligned = []
for i, s2_date in enumerate(s2_dates):
    ndvi_idx = find_nearest_ndvi(s2_date, ndvi_dates)
    julian   = date_to_julian(s2_date)
    dt       = datetime.strptime(s2_date, '%Y-%m-%d')
    aligned.append({
        's2_idx':   int(i),
        's1_idx':   int(i),
        'ndvi_idx': int(ndvi_idx),
        'date':     s2_date,
        'julian':   int(julian),
        'year':     int(dt.year),
        'month':    int(dt.month),
    })

print(f"Aligned timesteps: {len(aligned)}")
print(f"\nFirst 3 timesteps:")
for a in aligned[:3]:
    print(f"  {a['date']}  julian={a['julian']:3d}  "
          f"s2_idx={a['s2_idx']}  s1_idx={a['s1_idx']}  "
          f"ndvi_idx={a['ndvi_idx']}")

print(f"\nLast 3 timesteps:")
for a in aligned[-3:]:
    print(f"  {a['date']}  julian={a['julian']:3d}  "
          f"s2_idx={a['s2_idx']}  s1_idx={a['s1_idx']}  "
          f"ndvi_idx={a['ndvi_idx']}")

with open(str(PROCESSED_DIR / 'temporal_alignment.json'), 'w') as f:
    json.dump(aligned, f, indent=2)
print(f"\nSaved temporal_alignment.json")

Aligned timesteps: 137

First 3 timesteps:
  2019-01-01  julian=  1  s2_idx=0  s1_idx=0  ndvi_idx=0
  2019-01-17  julian= 17  s2_idx=1  s1_idx=1  ndvi_idx=1
  2019-02-02  julian= 33  s2_idx=2  s1_idx=2  ndvi_idx=1

Last 3 timesteps:
  2024-11-14  julian=319  s2_idx=134  s1_idx=134  ndvi_idx=70
  2024-11-30  julian=335  s2_idx=135  s1_idx=135  ndvi_idx=71
  2024-12-16  julian=351  s2_idx=136  s1_idx=136  ndvi_idx=71

Saved temporal_alignment.json


In [ ]:
##### Compute per-band normalization statistics from the training data

In [8]:
# Compute normalization stats from first 80% of timesteps (training set)
# Never compute stats from val/test — that would be data leakage
train_end = int(len(aligned) * 0.8)  # index 109

print(f"Computing normalization stats from timesteps 0-{train_end} (training set)")
print(f"Val/test timesteps: {train_end}-{len(aligned)} (held out)")

# S2 stats — per band
s2_train  = s2_arr[:train_end]   # (109, 14, 377, 313)
s2_mean   = s2_train.reshape(train_end, 14, -1).mean(axis=(0,2))
s2_std    = s2_train.reshape(train_end, 14, -1).std(axis=(0,2))
s2_std    = np.where(s2_std < 1e-6, 1.0, s2_std)  # avoid divide by zero

# S1 stats — per band
s1_train  = s1_arr[:train_end]   # (109, 2, 377, 313)
s1_mean   = s1_train.reshape(train_end, 2, -1).mean(axis=(0,2))
s1_std    = s1_train.reshape(train_end, 2, -1).std(axis=(0,2))

# CHIRPS stats
chirps_train = chirps_z[:train_end]  # (109, 4, 19, 16)
chirps_mean  = chirps_train.reshape(train_end, 4, -1).mean(axis=(0,2))
chirps_std   = chirps_train.reshape(train_end, 4, -1).std(axis=(0,2))
chirps_std   = np.where(chirps_std < 1e-6, 1.0, chirps_std)

S2_BANDS     = list(s2_z.attrs['bands'])
CHIRPS_BANDS = list(chirps_z.attrs['bands'])

print("\nS2 normalization stats (mean / std):")
for i, band in enumerate(S2_BANDS):
    print(f"  {band:<6}: mean={s2_mean[i]:>7.4f}  std={s2_std[i]:.4f}")

print("\nS1 normalization stats:")
for i, band in enumerate(['VV', 'VH']):
    print(f"  {band:<6}: mean={s1_mean[i]:>7.4f}  std={s1_std[i]:.4f}")

print("\nCHIRPS normalization stats:")
for i, band in enumerate(CHIRPS_BANDS):
    print(f"  {band:<10}: mean={chirps_mean[i]:>7.4f}  std={chirps_std[i]:.4f}")

# Save stats for use during inference
stats = {
    's2_mean':     s2_mean.tolist(),
    's2_std':      s2_std.tolist(),
    's1_mean':     s1_mean.tolist(),
    's1_std':      s1_std.tolist(),
    'chirps_mean': chirps_mean.tolist(),
    'chirps_std':  chirps_std.tolist(),
    'train_end':   int(train_end),
    's2_bands':    S2_BANDS,
    'chirps_bands':CHIRPS_BANDS,
}
with open(str(PROCESSED_DIR / 'normalization_stats.json'), 'w') as f:
    json.dump(stats, f, indent=2)
print(f"\nSaved normalization_stats.json")

Computing normalization stats from timesteps 0-109 (training set)
Val/test timesteps: 109-137 (held out)

S2 normalization stats (mean / std):
  B2    : mean= 0.0585  std=0.0312
  B3    : mean= 0.0852  std=0.0309
  B4    : mean= 0.0847  std=0.0383
  B5    : mean= 0.1325  std=0.0383
  B6    : mean= 0.2292  std=0.0520
  B7    : mean= 0.2606  std=0.0605
  B8    : mean= 0.2648  std=0.0601
  B8A   : mean= 0.2867  std=0.0643
  B11   : mean= 0.2578  std=0.0703
  B12   : mean= 0.1703  std=0.0649
  NDVI  : mean= 0.5078  std=0.1699
  EVI   : mean= 0.4231  std=312.1488
  NBR   : mean= 0.2249  std=0.1962
  NDWI  : mean=-0.5028  std=0.1221

S1 normalization stats:
  VV    : mean=-12.0695  std=2.0060
  VH    : mean=-18.2454  std=2.2569

CHIRPS normalization stats:
  rain_sum  : mean=57.4071  std=45.1505
  dry_days  : mean=11.2872  std=3.2215
  rain_mean : mean= 3.5879  std=2.8219
  rain_90d  : mean=383.8802  std=153.5829

Saved normalization_stats.json


In [ ]:
##### EVI std is 312.1488 which is wildly wrong — EVI should be in the range 0–1 like NDVI. 
#### This means there are outlier values in the EVI band from the 
###cloud masking. Fix it before normalizing

In [9]:
# Diagnose EVI outliers
evi_band = s2_arr[:, 11, :, :]  # EVI is band index 11
print("EVI distribution:")
print(f"  Min:    {evi_band.min():.3f}")
print(f"  Max:    {evi_band.max():.3f}")
print(f"  Mean:   {evi_band.mean():.3f}")
print(f"  Std:    {evi_band.std():.3f}")
print(f"  Values > 2:  {(evi_band > 2).sum():,}")
print(f"  Values < -1: {(evi_band < -1).sum():,}")

# Fix — clip EVI to physically valid range [-1, 2]
# then recompute std
print("\nClipping EVI to valid range [-1, 2]...")
s2_arr[:, 11, :, :] = np.clip(evi_band, -1.0, 2.0)

# Recompute S2 stats with clean EVI
s2_train  = s2_arr[:train_end]
s2_mean   = s2_train.reshape(train_end, 14, -1).mean(axis=(0,2))
s2_std    = s2_train.reshape(train_end, 14, -1).std(axis=(0,2))
s2_std    = np.where(s2_std < 1e-6, 1.0, s2_std)

print("\nCorrected S2 stats:")
for i, band in enumerate(S2_BANDS):
    print(f"  {band:<6}: mean={s2_mean[i]:>7.4f}  std={s2_std[i]:.4f}")

# Update saved stats
stats['s2_mean'] = s2_mean.tolist()
stats['s2_std']  = s2_std.tolist()
with open(str(PROCESSED_DIR / 'normalization_stats.json'), 'w') as f:
    json.dump(stats, f, indent=2)
print("\nUpdated normalization_stats.json")

EVI distribution:
  Min:    -86.275
  Max:    1119484.000
  Mean:   0.411
  Std:    278.429
  Values > 2:  196
  Values < -1: 302

Clipping EVI to valid range [-1, 2]...

Corrected S2 stats:
  B2    : mean= 0.0585  std=0.0312
  B3    : mean= 0.0852  std=0.0309
  B4    : mean= 0.0847  std=0.0383
  B5    : mean= 0.1325  std=0.0383
  B6    : mean= 0.2292  std=0.0520
  B7    : mean= 0.2606  std=0.0605
  B8    : mean= 0.2648  std=0.0601
  B8A   : mean= 0.2867  std=0.0643
  B11   : mean= 0.2578  std=0.0703
  B12   : mean= 0.1703  std=0.0649
  NDVI  : mean= 0.5078  std=0.1699
  EVI   : mean= 0.3361  std=0.1212
  NBR   : mean= 0.2249  std=0.1962
  NDWI  : mean=-0.5028  std=0.1221

Updated normalization_stats.json


In [ ]:
#### Normalize all arrays and extract patches

In [10]:
# Normalize all arrays using training stats
print("Normalizing arrays...")

def normalize(arr, mean, std):
    """Z-score normalize along band dimension."""
    return (arr - mean[None, :, None, None]) / std[None, :, None, None]

s2_norm     = normalize(s2_arr,            s2_mean,     s2_std)
s1_norm     = normalize(s1_arr,            s1_mean,     s1_std)
chirps_full = chirps_z[:]                # (137, 4, 19, 16)
chirps_norm = normalize(chirps_full,      chirps_mean, chirps_std)

print(f"S2 normalized:     mean={s2_norm.mean():.4f}  std={s2_norm.std():.4f}")
print(f"S1 normalized:     mean={s1_norm.mean():.4f}  std={s1_norm.std():.4f}")
print(f"CHIRPS normalized: mean={chirps_norm.mean():.4f}  std={chirps_norm.std():.4f}")

# Extract 64x64 patches with 32px stride (50% overlap)
PATCH_SIZE = 64
STRIDE     = 32

def extract_patches(arr_s2, arr_s1, arr_chirps, aligned,
                    patch_size=64, stride=32):
    """
    Extract spatial patches across the full time series.
    
    For each (row, col) position:
      - Extract 64x64 spatial patch from S2 and S1
      - Assign nearest CHIRPS pixel (5km resolution)
      - Stack all 137 timesteps as the temporal dimension
    
    Returns list of patch dicts ready for PyTorch Dataset.
    """
    T, _, H, W = arr_s2.shape
    patches    = []
    rows = range(0, H - patch_size + 1, stride)
    cols = range(0, W - patch_size + 1, stride)

    print(f"Grid: {len(rows)} rows × {len(cols)} cols = "
          f"{len(rows)*len(cols)} patches")

    for r in tqdm(rows, desc='Extracting patches'):
        for c in cols:
            # S2 patch: (137, 14, 64, 64)
            s2_patch = arr_s2[:, :, r:r+patch_size, c:c+patch_size]

            # S1 patch: (137, 2, 64, 64)
            s1_patch = arr_s1[:, :, r:r+patch_size, c:c+patch_size]

            # CHIRPS: map S2 pixel to nearest CHIRPS pixel
            # CHIRPS is 19x16 vs S2 377x313 — scale factor ~20x
            chirps_r = min(int(r / H * arr_chirps.shape[2]),
                           arr_chirps.shape[2] - 1)
            chirps_c = min(int(c / W * arr_chirps.shape[3]),
                           arr_chirps.shape[3] - 1)
            # CHIRPS pixel time series: (137, 4)
            chirps_pixel = arr_chirps[:, :, chirps_r, chirps_c]

            # Julian day array: (137,)
            julian_days = np.array([a['julian'] for a in aligned],
                                   dtype=np.float32)

            patches.append({
                'row':         r,
                'col':         c,
                's2':          s2_patch.astype(np.float32),
                's1':          s1_patch.astype(np.float32),
                'chirps':      chirps_pixel.astype(np.float32),
                'julian_days': julian_days,
            })

    return patches

patches = extract_patches(s2_norm, s1_norm, chirps_norm, aligned)
print(f"\nTotal patches extracted: {len(patches)}")
print(f"Per patch shapes:")
print(f"  s2:          {patches[0]['s2'].shape}")
print(f"  s1:          {patches[0]['s1'].shape}")
print(f"  chirps:      {patches[0]['chirps'].shape}")
print(f"  julian_days: {patches[0]['julian_days'].shape}")

Normalizing arrays...
S2 normalized:     mean=-0.0090  std=1.0397
S1 normalized:     mean=0.0508  std=0.9868
CHIRPS normalized: mean=0.0093  std=1.0106
Grid: 10 rows × 8 cols = 80 patches


Extracting patches: 100%|██████████████████████████████████████████████████████████████| 10/10 [00:02<00:00,  3.36it/s]


Total patches extracted: 80
Per patch shapes:
  s2:          (137, 14, 64, 64)
  s1:          (137, 2, 64, 64)
  chirps:      (137, 4)
  julian_days: (137,)


In [ ]:
##### 80 patches extracted with correct shapes. Now split into train/val/test and save as a PyTorch Dataset:

In [11]:
# Train/val/test split — split by time not space
# Train: timesteps 0-109, Val: 110-123, Test: 124-136
TRAIN_END = 109
VAL_END   = 123

print(f"Temporal split:")
print(f"  Train: timesteps 0-{TRAIN_END}   "
      f"({s2_dates[0]} to {s2_dates[TRAIN_END]})")
print(f"  Val:   timesteps {TRAIN_END}-{VAL_END}  "
      f"({s2_dates[TRAIN_END]} to {s2_dates[VAL_END]})")
print(f"  Test:  timesteps {VAL_END}-136   "
      f"({s2_dates[VAL_END]} to {s2_dates[-1]})")

def split_patch(patch, train_end, val_end):
    """Split a patch dict into train/val/test along time axis."""
    def slice_patch(start, end):
        return {
            'row':         patch['row'],
            'col':         patch['col'],
            's2':          patch['s2'][start:end],
            's1':          patch['s1'][start:end],
            'chirps':      patch['chirps'][start:end],
            'julian_days': patch['julian_days'][start:end],
        }
    return (slice_patch(0,         train_end),
            slice_patch(train_end, val_end),
            slice_patch(val_end,   None))

train_patches, val_patches, test_patches = [], [], []
for p in patches:
    tr, va, te = split_patch(p, TRAIN_END, VAL_END)
    train_patches.append(tr)
    val_patches.append(va)
    test_patches.append(te)

print(f"\nPatches per split:")
print(f"  Train: {len(train_patches)} patches × "
      f"{train_patches[0]['s2'].shape[0]} timesteps")
print(f"  Val:   {len(val_patches)} patches × "
      f"{val_patches[0]['s2'].shape[0]} timesteps")
print(f"  Test:  {len(test_patches)} patches × "
      f"{test_patches[0]['s2'].shape[0]} timesteps")

# PyTorch Dataset
class MaraDataset(Dataset):
    """
    PyTorch Dataset for VegFormer training.
    Each item is one spatial patch across all timesteps.
    
    Returns dict with:
        s2:          (T, 14, 64, 64) optical bands
        s1:          (T,  2, 64, 64) SAR bands
        chirps:      (T,  4)         climate variables
        julian_days: (T,)            day of year
        row, col:    spatial position
    """
    def __init__(self, patches):
        self.patches = patches

    def __len__(self):
        return len(self.patches)

    def __getitem__(self, idx):
        p = self.patches[idx]
        return {
            's2':          torch.tensor(p['s2'],          dtype=torch.float32),
            's1':          torch.tensor(p['s1'],          dtype=torch.float32),
            'chirps':      torch.tensor(p['chirps'],      dtype=torch.float32),
            'julian_days': torch.tensor(p['julian_days'], dtype=torch.float32),
            'row':         torch.tensor(p['row'],         dtype=torch.long),
            'col':         torch.tensor(p['col'],         dtype=torch.long),
        }

train_ds = MaraDataset(train_patches)
val_ds   = MaraDataset(val_patches)
test_ds  = MaraDataset(test_patches)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=4, shuffle=False, num_workers=0)

# Test one batch
batch = next(iter(train_loader))
print(f"\nSample batch shapes:")
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:<12}: {tuple(v.shape)}")

print(f"\nDataLoaders ready:")
print(f"  Train: {len(train_loader)} batches")
print(f"  Val:   {len(val_loader)} batches")
print(f"  Test:  {len(test_loader)} batches")

Temporal split:
  Train: timesteps 0-109   (2019-01-01 to 2023-10-11)
  Val:   timesteps 109-123  (2023-10-11 to 2024-05-22)
  Test:  timesteps 123-136   (2024-05-22 to 2024-12-16)

Patches per split:
  Train: 80 patches × 109 timesteps
  Val:   80 patches × 14 timesteps
  Test:  80 patches × 14 timesteps

Sample batch shapes:
  s2          : (4, 109, 14, 64, 64)
  s1          : (4, 109, 2, 64, 64)
  chirps      : (4, 109, 4)
  julian_days : (4, 109)
  row         : (4,)
  col         : (4,)

DataLoaders ready:
  Train: 20 batches
  Val:   20 batches
  Test:  20 batches


In [ ]:
##### Now save the patches and run a smoke test of the VegFormer model on one batch:

In [15]:
# Fix vegformer.py — update assertion and positional encoding dim
vegformer_path = 'C:/vegformer/models/vegformer.py'

with open(vegformer_path, 'r') as f:
    code = f.read()

# Fix 1: update assertion to allow climate encoder contribution
code = code.replace(
    'assert 128 + 256 == d_model, "Encoder dims must sum to d_model"',
    '# d_model = sar_dim + optical_dim (climate added as residual after fusion)'
)

# Fix 2: fix positional encoding n_fourier default
# SeasonalPositionalEncoding(d_model, n_fourier=4) but project is Linear(4, d_model)
# n_fourier=4 means 2 harmonics × 2 (sin/cos) = 4 features — correct
# But build_vegformer passes d_model=384, pos_enc projects 4 -> 384 — should work
# Real issue: n_fourier param in __init__ vs actual features generated
code = code.replace(
    'in_dim = n_fourier * 2  # sin/cos for annual and semi-annual',
    'in_dim = n_fourier      # each harmonic already generates sin+cos pair'
)

with open(vegformer_path, 'w') as f:
    f.write(code)

print("vegformer.py patched")

# Reload the module
import importlib
import sys
if 'models.vegformer' in sys.modules:
    del sys.modules['models.vegformer']
if 'vegformer' in sys.modules:
    del sys.modules['vegformer']

from models.vegformer import build_vegformer

# Rebuild model
model = build_vegformer({
    'n_bands':        14,
    'n_climate_vars': 4,
    'd_model':        384,
    'n_heads':        8,
    'n_layers':       4,
    'n_veg_classes':  8,
    'n_deg_classes':  4,
    'dropout':        0.1,
    'max_seq_len':    137,
}).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"VegFormer parameters: {n_params:,}")

# Test forward pass
model.eval()
batch = next(iter(train_loader))

with torch.no_grad():
    model_input = {
        'sar':     batch['s1'].mean(dim=(-2,-1)).to(device),
        'optical': batch['s2'].mean(dim=(-2,-1)).to(device),
        'climate': batch['chirps'].to(device),
        'dates':   batch['julian_days'].to(device),
    }
    out = model(model_input)

print(f"\nVegFormer output shapes:")
for k, v in out.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:<12}: {tuple(v.shape)}")

print(f"\nSanity checks:")
print(f"  Risk scores in [0,1]: "
      f"{bool((out['risk_score'] >= 0).all() and (out['risk_score'] <= 1).all())}")
print(f"  No NaN in outputs:    "
      f"{not any(torch.isnan(v).any() for v in out.values() if isinstance(v, torch.Tensor))}")

vegformer.py patched
VegFormer parameters: 8,919,393


C:\vegformer\models\vegformer.py:120: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.temporal_attn = nn.TransformerEncoder(encoder_layer, num_layers=2)
C:\vegformer\models\vegformer.py:148: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
C:\vegformer\models\vegformer.py:296: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)



VegFormer output shapes:
  veg_type    : (4, 8)
  degradation : (4, 4)
  risk_score  : (4, 1)
  embeddings  : (4, 384)
  sar_attn    : (4, 109, 109)
  opt_attn    : (4, 109, 109)
  token_feats : (4, 109, 384)

Sanity checks:
  Risk scores in [0,1]: True
  No NaN in outputs:    True


In [16]:
# Save patches to disk
import pickle

for split, data in [('train', train_patches),
                    ('val',   val_patches),
                    ('test',  test_patches)]:
    path = PATCHES_DIR / f'{split}_patches.pkl'
    with open(path, 'wb') as f:
        pickle.dump(data, f)
    size = path.stat().st_size / 1e6
    print(f"Saved {split}_patches.pkl — "
          f"{len(data)} patches | {size:.1f} MB")

# Save dataset stats
dataset_info = {
    'n_patches':     len(patches),
    'patch_size':    PATCH_SIZE,
    'stride':        STRIDE,
    'train_patches': len(train_patches),
    'val_patches':   len(val_patches),
    'test_patches':  len(test_patches),
    'train_end':     TRAIN_END,
    'val_end':       VAL_END,
    'T_train':       int(train_patches[0]['s2'].shape[0]),
    'T_val':         int(val_patches[0]['s2'].shape[0]),
    'T_test':        int(test_patches[0]['s2'].shape[0]),
    'spatial_shape': [377, 313],
    'n_s2_bands':    14,
    'n_s1_bands':    2,
    'n_chirps_vars': 4,
}
with open(str(PROCESSED_DIR / 'dataset_info.json'), 'w') as f:
    json.dump(dataset_info, f, indent=2)
print(f"\nSaved dataset_info.json")

# Smoke test VegFormer on one real batch
print("\nRunning VegFormer smoke test on real data...")
import sys
sys.path.append('C:/vegformer')
from models.vegformer import build_vegformer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

model = build_vegformer({
    'n_bands':        14,
    'n_climate_vars': 4,
    'd_model':        448,
    'n_heads':        8,
    'n_layers':       4,
    'n_veg_classes':  8,
    'n_deg_classes':  4,
    'dropout':        0.1,
    'max_seq_len':    137,
}).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"VegFormer parameters: {n_params:,}")

# Run one batch through the model
model.eval()
batch = next(iter(train_loader))

# VegFormer expects (B, T, C) for SAR and climate
# and (B, T, C, H, W) is too large — use spatial mean for now
# Full spatial attention comes in notebook 04
with torch.no_grad():
    model_input = {
        'sar':     batch['s1'].mean(dim=(-2,-1)).to(device),  # (B, T, 2)
        'optical': batch['s2'].mean(dim=(-2,-1)).to(device),  # (B, T, 14)
        'climate': batch['chirps'].to(device),                 # (B, T, 4)
        'dates':   batch['julian_days'].to(device),            # (B, T)
    }
    out = model(model_input)

print(f"\nVegFormer output shapes:")
for k, v in out.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:<12}: {tuple(v.shape)}")

print(f"\nSanity checks:")
print(f"  Risk scores in [0,1]: "
      f"{(out['risk_score'] >= 0).all() and (out['risk_score'] <= 1).all()}")
print(f"  No NaN in outputs:    "
      f"{not any(torch.isnan(v).any() for v in out.values() if isinstance(v, torch.Tensor))}")

Saved train_patches.pkl — 80 patches | 2286.1 MB
Saved val_patches.pkl — 80 patches | 293.6 MB
Saved test_patches.pkl — 80 patches | 293.6 MB

Saved dataset_info.json

Running VegFormer smoke test on real data...
Device: cpu
VegFormer parameters: 11,538,209


RuntimeError: The size of tensor a (384) must match the size of tensor b (448) at non-singleton dimension 2

In [17]:
model = build_vegformer({
    'n_bands':        14,
    'n_climate_vars': 4,
    'd_model':        384,   # 128 (SAR) + 256 (optical) = 384
    'n_heads':        8,
    'n_layers':       4,
    'n_veg_classes':  8,
    'n_deg_classes':  4,
    'dropout':        0.1,
    'max_seq_len':    137,
}).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Device:               {device}")
print(f"VegFormer parameters: {n_params:,}")

# Run one batch through the model
model.eval()
batch = next(iter(train_loader))

with torch.no_grad():
    model_input = {
        'sar':     batch['s1'].mean(dim=(-2,-1)).to(device),
        'optical': batch['s2'].mean(dim=(-2,-1)).to(device),
        'climate': batch['chirps'].to(device),
        'dates':   batch['julian_days'].to(device),
    }
    out = model(model_input)

print(f"\nVegFormer output shapes:")
for k, v in out.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:<12}: {tuple(v.shape)}")

print(f"\nSanity checks:")
print(f"  Risk scores in [0,1]: "
      f"{bool((out['risk_score'] >= 0).all() and (out['risk_score'] <= 1).all())}")
print(f"  No NaN in outputs:    "
      f"{not any(torch.isnan(v).any() for v in out.values() if isinstance(v, torch.Tensor))}")

Device:               cpu
VegFormer parameters: 8,919,393

VegFormer output shapes:
  veg_type    : (4, 8)
  degradation : (4, 4)
  risk_score  : (4, 1)
  embeddings  : (4, 384)
  sar_attn    : (4, 109, 109)
  opt_attn    : (4, 109, 109)
  token_feats : (4, 109, 384)

Sanity checks:
  Risk scores in [0,1]: True
  No NaN in outputs:    True
